# Brain Tumor Classification — Xception (Transfer Learning)

Task 1: Train the Xception model to classify Brain MRI scans into 4 classes:
- glioma
- meningioma
- no_tumor
- pituitary

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import Xception
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 1. Configuration

In [ ]:
DATA_DIR = '../data/brain_tumor_dataset'
TRAIN_DIR = os.path.join(DATA_DIR, 'Training')
TEST_DIR  = os.path.join(DATA_DIR, 'Testing')

IMAGE_SIZE  = (299, 299)  # Xception native input size
BATCH_SIZE  = 32
EPOCHS_FROZEN  = 10      # train only top layers first
EPOCHS_FINETUNE = 20     # then unfreeze and fine-tune
CLASSES = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
NUM_CLASSES = len(CLASSES)
MODEL_SAVE_PATH = '../models/xception_brain_tumor.keras'

## 2. Data Loading & Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.15,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42,
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

print('Class indices:', train_generator.class_indices)

## 3. Visualize Sample Images

In [ ]:
images, labels = next(train_generator)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(CLASSES[np.argmax(labels[i])])
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Build the Xception Model

In [ ]:
def build_xception_model(num_classes: int, image_size: tuple) -> keras.Model:
    base_model = Xception(
        weights='imagenet',
        include_top=False,
        input_shape=(*image_size, 3),
    )
    base_model.trainable = False  # freeze for initial training

    inputs = keras.Input(shape=(*image_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs), base_model


model, base_model = build_xception_model(NUM_CLASSES, IMAGE_SIZE)
model.summary()

## 5. Phase 1 — Train Top Layers (base frozen)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6),
]

history_frozen = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FROZEN,
    callbacks=callbacks,
)

## 6. Phase 2 — Fine-tune (unfreeze top layers of base)

In [ ]:
# Unfreeze the top 30 layers of Xception
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINETUNE,
    callbacks=callbacks,
)

model.save(MODEL_SAVE_PATH)
print('Model saved to', MODEL_SAVE_PATH)

## 7. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f'Test accuracy: {test_acc:.4f}')

y_pred = np.argmax(model.predict(test_generator), axis=1)
y_true = test_generator.classes

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.title('Confusion Matrix — Xception')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 8. Training History Plots

In [ ]:
def plot_history(h1, h2=None, title='Training History'):
    acc  = h1.history['accuracy']
    val  = h1.history['val_accuracy']
    loss = h1.history['loss']
    vloss = h1.history['val_loss']
    if h2:
        acc  += h2.history['accuracy']
        val  += h2.history['val_accuracy']
        loss += h2.history['loss']
        vloss += h2.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, acc, label='Train Acc')
    ax1.plot(epochs, val, label='Val Acc')
    if h2:
        ax1.axvline(len(h1.history['accuracy']), color='gray', linestyle='--', label='Fine-tune start')
    ax1.set_title('Accuracy'); ax1.legend()

    ax2.plot(epochs, loss, label='Train Loss')
    ax2.plot(epochs, vloss, label='Val Loss')
    if h2:
        ax2.axvline(len(h1.history['loss']), color='gray', linestyle='--', label='Fine-tune start')
    ax2.set_title('Loss'); ax2.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_history(history_frozen, history_finetune, 'Xception Training History')